In [ ]:
import pyspark.sql.functions as F

In [ ]:
catalog =dbutils.widgets.get("catalog")

In [ ]:
matches = spark.table(f"{catalog}.bronze.raw_matches_tbl")
matches = matches.select(
            F.col("competition.id").alias("competition_id"),
            F.col("competition.name").alias("competition"),
            F.explode("matches").alias("match"),
        )
matches = matches.select(
            "competition_id",
            "competition",
            F.col("match.id").alias("match_id"),
            F.col("match.homeTeam.id").alias("home_team_id"),
            F.col("match.homeTeam.name").alias("home_team"),
            F.col("match.awayTeam.id").alias("away_team_id"),
            F.col("match.awayTeam.name").alias("away_team"),
            F.col("match.score.duration").alias("duration"),
            F.col("match.score.extraTime").alias("extra_time"),
            F.col("match.score.fullTime.home").alias("home_team_score"),
            F.col("match.score.fullTime.away").alias("away_team_score"),
            F.col("match.score.winner").alias("winner"))
matches.write.format("delta").mode("overwrite").saveAsTable(f"{catalog}.silver.transformed_matches_tbl")

In [ ]:
scorers_df = spark.table(f"{catalog}.bronze.raw_scorers_tbl")
scorers_df = scorers_df.select(
    F.col("competition.id").alias("competition_id"),
    F.col("competition.name").alias("competition"),
    F.explode("scorers").alias("scorer")
)
scorers_df = scorers_df.select(
            "competition_id",
            "competition",
            F.col("scorer.player.name").alias("player_name"),
            F.col("scorer.player.id").alias("player_id"),
            F.col("scorer.team.name").alias("team_name"),
            F.col("scorer.team.id").alias("team_id"),
            F.col("scorer.player.nationality").alias("nationality"),
            F.col("scorer.player.position").alias("position"),
            F.col("scorer.player.dateOfBirth").alias("date_of_birth"),
            F.col("scorer.assists").alias("assists"),
            F.col("scorer.goals").alias("goals"),
            F.col("scorer.penalties").alias("penalties"),
            F.col("scorer.playedMatches").alias("played_matches"),
        )
scorers_df.write.format("delta").mode("overwrite").saveAsTable(f"{catalog}.silver.transformed_scorers_tbl")

In [ ]:
standings_df = spark.table(f"{catalog}.bronze.raw_standings_tbl")
standings_df = standings_df.select(
    F.col("competition.id").alias("competition_id"),
    F.col("competition.name").alias("competition"),
    F.explode("standings").alias("standing"))
standings_df = standings_df.select(
    "competition_id",
    "competition",
    F.explode("standing.table").alias("table"))
standings_df = standings_df.select(
    "competition_id",
    "competition",
    F.col('table.team.id').alias('team_id'),
    F.col('table.team.name').alias('team_name'), 
    F.col('table.position').alias('position'), 
    F.col('table.points').alias('points'), 
    F.col('table.playedGames').alias('games'),
    F.col('table.won').alias('wins'),
    F.col('table.draw').alias('draws'),
    F.col('table.lost').alias('losses'),
    F.col('table.goalsFor').alias('goals_for'),
    F.col('table.goalsAgainst').alias('goals_against'),
    F.col('table.goalDifference').alias('goal_difference')
)
standings_df.write.format("delta").mode("overwrite").saveAsTable(f"{catalog}.silver.transformed_standings_tbl")

In [ ]:
teams = spark.table(f"{catalog}.bronze.raw_teams_tbl")
teams = teams.select(
    F.col("competition.id").alias("competition_id"),
    F.col("competition.name").alias("competition"),
    F.explode("teams").alias("team")
)
teams = teams.select(
    "competition_id",
    "competition",
    F.col("team.id").alias("team_id"),
            F.col("team.name").alias("team_name"),
            F.col("team.coach.name").alias("coach"),
            F.col("team.coach.contract.start").alias("contract_start"),
            F.col("team.coach.contract.until").alias("contract_end"),
            F.col("team.lastUpdated").alias("last_updated"),
            F.col("team.squad").alias("squad")
)
teams.write.format("delta").mode("overwrite").saveAsTable(f"{catalog}.silver.transformed_teams_tbl")